In [3]:
# Celda 1 - PB-07 / PB-08 / PB-09: librerías y configuración inicial
# Objetivo: preparar el entorno para integrar limpieza, feature engineering,
# balanceo y pipeline reproducible del Sprint 2

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

In [4]:
# Celda 2 - PB-07 / PB-08 / PB-09: rutas y carga del dataset crudo
# Objetivo: partir desde data/raw para integrar el flujo completo del Sprint 2

DATA_PATH = "../data/raw/dataset_pucp.csv"

df_raw = pd.read_csv(DATA_PATH)

print("Dataset crudo cargado correctamente.")
print("Dimensiones:", df_raw.shape)
display(df_raw.head())

Dataset crudo cargado correctamente.
Dimensiones: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [5]:
# Celda 3 - PB-07: revisión mínima y limpieza base
# Objetivo: partir de una versión limpia del dataset para integrarla al pipeline

print("Dimensiones originales:", df_raw.shape)
print("Duplicados detectados:", df_raw.duplicated().sum())

df_clean = df_raw.drop_duplicates().copy()

print("Dimensiones después de eliminar duplicados:", df_clean.shape)

print("\nValores nulos por columna:")
display(df_clean.isnull().sum().to_frame("nulos"))

Dimensiones originales: (12330, 18)
Duplicados detectados: 125
Dimensiones después de eliminar duplicados: (12205, 18)

Valores nulos por columna:


,nulos
Administrative,0
Administrative_Duration,0
Informational,0
Informational_Duration,0
ProductRelated,0
ProductRelated_Duration,0
BounceRates,0
ExitRates,0
PageValues,0
SpecialDay,0


In [6]:
# Celda 4 - PB-07: ajuste de tipos y definición de columnas
# Objetivo: dejar explícito qué columnas serán tratadas como numéricas,
# categóricas y booleanas dentro del pipeline

df_prep = df_clean.copy()

target_col = "Revenue"

num_cols = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

cat_cols = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType"
]

bool_cols = [
    "Weekend"
]

# Ajuste de tipos
for col in cat_cols:
    df_prep[col] = df_prep[col].astype(str)

for col in bool_cols:
    df_prep[col] = df_prep[col].astype(int)

print("Variable objetivo:", target_col)
print("\nColumnas numéricas:", num_cols)
print("\nColumnas categóricas:", cat_cols)
print("\nColumnas booleanas:", bool_cols)

print("\nTipos de datos ajustados:")
display(df_prep[cat_cols + bool_cols + [target_col]].dtypes.to_frame("tipo"))

Variable objetivo: Revenue

Columnas numéricas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

Columnas categóricas: ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']

Columnas booleanas: ['Weekend']

Tipos de datos ajustados:


,tipo
Month,str
OperatingSystems,str
Browser,str
Region,str
TrafficType,str
VisitorType,str
Weekend,int64
Revenue,bool


In [7]:
# Celda 5 - PB-08: creación de features derivadas seleccionadas
# Objetivo: incorporar pocas variables nuevas, simples y defendibles para el pipeline final

df_feat = df_prep.copy()

df_feat["total_paginas"] = (
    df_feat["Administrative"] +
    df_feat["Informational"] +
    df_feat["ProductRelated"]
)

df_feat["duracion_total"] = (
    df_feat["Administrative_Duration"] +
    df_feat["Informational_Duration"] +
    df_feat["ProductRelated_Duration"]
)

print("Nuevas features creadas correctamente.")
display(df_feat[["total_paginas", "duracion_total"]].describe().T)

Nuevas features creadas correctamente.


,count,mean,std,min,25%,50%,75%,max
total_paginas,12205.0,34.893240,46.627336,0.0,9.000000,20.000000,42.000000,746.00000
duracion_total,12205.0,1323.454242,2043.871589,0.0,231.666667,690.958333,1643.958333,69921.64723


In [8]:
# Celda 6 - PB-08: documentación breve de features creadas
# Objetivo: dejar evidencia clara de qué features nuevas se incorporan y por qué

feature_docs = pd.DataFrame([
    {
        "feature": "total_paginas",
        "como_se_calcula": "Administrative + Informational + ProductRelated",
        "por_que_podria_ser_util": "Resume el volumen total de navegación del usuario dentro del sitio."
    },
    {
        "feature": "duracion_total",
        "como_se_calcula": "Administrative_Duration + Informational_Duration + ProductRelated_Duration",
        "por_que_podria_ser_util": "Resume el tiempo total de navegación y puede reflejar mayor interés o profundidad de sesión."
    }
])

display(feature_docs)

,feature,como_se_calcula,por_que_podria_ser_util
0,total_paginas,Administrative + Informational + ProductRelated,Resume el volumen total de navegación del usua...
1,duracion_total,Administrative_Duration + Informational_Durati...,Resume el tiempo total de navegación y puede r...


In [9]:
# Celda 7 - PB-09: definición de X e y y train/test split estratificado
# Objetivo: separar entrenamiento y prueba antes de cualquier fit del pipeline

X = df_feat.drop(columns=[target_col]).copy()
y = df_feat[target_col].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test :", X_test.shape)
print("Dimensiones de y_train:", y_train.shape)
print("Dimensiones de y_test :", y_test.shape)

print("\nDistribución de clases en y_train:")
display(y_train.value_counts().to_frame("frecuencia"))
display((y_train.value_counts(normalize=True) * 100).round(2).to_frame("%"))

print("\nDistribución de clases en y_test:")
display(y_test.value_counts().to_frame("frecuencia"))
display((y_test.value_counts(normalize=True) * 100).round(2).to_frame("%"))

Dimensiones de X_train: (9764, 19)
Dimensiones de X_test : (2441, 19)
Dimensiones de y_train: (9764,)
Dimensiones de y_test : (2441,)

Distribución de clases en y_train:


,frecuencia
Revenue,
0,8238
1,1526


,%
Revenue,
0,84.37
1,15.63



Distribución de clases en y_test:


,frecuencia
Revenue,
0,2059
1,382


,%
Revenue,
0,84.35
1,15.65


In [10]:
# Celda 8 - PB-07 / PB-08: definición final de columnas para el preprocesador
# Objetivo: incluir también las features nuevas dentro del grupo numérico del pipeline

num_cols_model = num_cols + ["total_paginas", "duracion_total"]
cat_cols_model = cat_cols
bool_cols_model = bool_cols

print("Columnas numéricas para el pipeline:")
print(num_cols_model)

print("\nColumnas categóricas para el pipeline:")
print(cat_cols_model)

print("\nColumnas booleanas para el pipeline:")
print(bool_cols_model)

Columnas numéricas para el pipeline:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'total_paginas', 'duracion_total']

Columnas categóricas para el pipeline:
['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']

Columnas booleanas para el pipeline:
['Weekend']


In [11]:
# Celda 9 - PB-07 / PB-08: construcción del preprocesador
# Objetivo: aplicar transformaciones diferenciadas por tipo de variable

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_model),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_model),
        ("bool", "passthrough", bool_cols_model)
    ]
)

print("Preprocesador creado correctamente.")
print(preprocessor)

Preprocesador creado correctamente.
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Administrative', 'Administrative_Duration',
                                  'Informational', 'Informational_Duration',
                                  'ProductRelated', 'ProductRelated_Duration',
                                  'BounceRates', 'ExitRates', 'PageValues',
                                  'SpecialDay', 'total_paginas',
                                  'duracion_total']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Month', 'OperatingSystems', 'Browser',
                                  'Region', 'TrafficType', 'VisitorType']),
                                ('bool', 'passthrough', ['Weekend'])])


In [12]:
# Celda 10 - PB-09: construcción del pipeline con balanceo
# Objetivo: integrar preprocesamiento y SMOTE en un flujo reproducible

pipeline_preprocessing = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42))
])

print("Pipeline creado correctamente.")
print(pipeline_preprocessing)

Pipeline creado correctamente.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'BounceRates', 'ExitRates',
                                                   'PageValues', 'SpecialDay',
                                                   'total_paginas',
                                                   'duracion_total']),
                                                 ('cat',
                                                  OneHotEncoder(hand

In [13]:
# Celda 11 - PB-07 / PB-08 / PB-09: validación funcional del pipeline
# Objetivo: comprobar que el pipeline puede ajustar el preprocesamiento
# y balancear correctamente el conjunto de entrenamiento

X_train_resampled, y_train_resampled = pipeline_preprocessing.fit_resample(X_train, y_train)

print("Forma de X_train después del pipeline:", X_train_resampled.shape)
print("Forma de y_train después del pipeline:", y_train_resampled.shape)

print("\nDistribución de clases luego del balanceo:")
display(pd.Series(y_train_resampled).value_counts().to_frame("frecuencia"))
display((pd.Series(y_train_resampled).value_counts(normalize=True) * 100).round(2).to_frame("%"))

Forma de X_train después del pipeline: (16476, 75)
Forma de y_train después del pipeline: (16476,)

Distribución de clases luego del balanceo:


,frecuencia
Revenue,
1,8238
0,8238


,%
Revenue,
1,50.0
0,50.0


In [14]:
# Celda 12 - PB-07 / PB-08 / PB-09: transformación del conjunto de prueba
# Objetivo: aplicar al test solo el preprocesamiento, sin balancearlo

X_test_processed = pipeline_preprocessing.named_steps["preprocessor"].transform(X_test)

print("Forma de X_test después del preprocesamiento:", X_test_processed.shape)

Forma de X_test después del preprocesamiento: (2441, 75)


In [15]:
# Celda 13 - PB-07 / PB-08 / PB-09: generación de DataFrames procesados
# Objetivo: convertir train y test procesados en tablas con nombres de columnas

from scipy import sparse

feature_names = pipeline_preprocessing.named_steps["preprocessor"].get_feature_names_out()

# Convertir matrices sparse a dense si es necesario
if sparse.issparse(X_train_resampled):
    X_train_array = X_train_resampled.toarray()
else:
    X_train_array = X_train_resampled

if sparse.issparse(X_test_processed):
    X_test_array = X_test_processed.toarray()
else:
    X_test_array = X_test_processed

df_train_processed = pd.DataFrame(X_train_array, columns=feature_names)
df_train_processed["Revenue"] = y_train_resampled

df_test_processed = pd.DataFrame(X_test_array, columns=feature_names)
df_test_processed["Revenue"] = y_test.reset_index(drop=True)

print("Dimensiones train procesado:", df_train_processed.shape)
print("Dimensiones test procesado :", df_test_processed.shape)

display(df_train_processed.head())
display(df_test_processed.head())

Dimensiones train procesado: (16476, 76)
Dimensiones test procesado : (2441, 76)


,num__Administrative,num__Administrative_Duration,num__Informational,num__Informational_Duration,num__ProductRelated,num__ProductRelated_Duration,num__BounceRates,num__ExitRates,num__PageValues,num__SpecialDay,...,cat__TrafficType_5,cat__TrafficType_6,cat__TrafficType_7,cat__TrafficType_8,cat__TrafficType_9,cat__VisitorType_New_Visitor,cat__VisitorType_Other,cat__VisitorType_Returning_Visitor,bool__Weekend,Revenue
0,1.689206,0.676238,0.381750,-0.243015,0.220868,-0.080395,-0.448703,-0.800192,1.665596,-0.311837,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
1,-0.699498,-0.458067,-0.397879,-0.243015,-0.046900,-0.158842,-0.448703,-0.739774,1.495448,-0.311837,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
2,-0.699498,-0.458067,-0.397879,-0.243015,-0.426237,-0.408089,-0.448703,-0.287446,-0.315264,3.689185,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
3,-0.102322,-0.235218,-0.397879,-0.243015,-0.604748,-0.336795,-0.448703,-0.279507,-0.315264,-0.311837,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0
4,-0.699498,-0.458067,-0.397879,-0.243015,-0.359295,0.203501,-0.448703,-0.609743,-0.315264,-0.311837,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0


,num__Administrative,num__Administrative_Duration,num__Informational,num__Informational_Duration,num__ProductRelated,num__ProductRelated_Duration,num__BounceRates,num__ExitRates,num__PageValues,num__SpecialDay,...,cat__TrafficType_5,cat__TrafficType_6,cat__TrafficType_7,cat__TrafficType_8,cat__TrafficType_9,cat__VisitorType_New_Visitor,cat__VisitorType_Other,cat__VisitorType_Returning_Visitor,bool__Weekend,Revenue
0,-0.699498,-0.458067,-0.397879,-0.243015,-0.448551,-0.376249,-0.006534,-0.031831,-0.315264,-0.311837,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
1,-0.102322,1.141994,-0.397879,-0.243015,-0.359295,-0.381034,-0.448703,-0.850540,-0.315264,-0.311837,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0
2,-0.102322,0.588769,2.720635,0.452989,-0.069213,0.467099,-0.448703,-0.373324,-0.315264,-0.311837,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
3,1.390618,1.103366,-0.397879,-0.243015,-0.470865,-0.455581,-0.448703,-0.398583,-0.315264,-0.311837,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4,-0.699498,-0.458067,-0.397879,-0.243015,-0.604748,-0.626730,3.972987,3.435644,-0.315264,-0.311837,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0


In [16]:
# Celda 14 - PB-07 / PB-08 / PB-09: guardado de datasets procesados
# Objetivo: exportar los outputs del Sprint 2 a data/processed

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df_train_processed.to_csv(processed_dir / "train_processed.csv", index=False)
df_test_processed.to_csv(processed_dir / "test_processed.csv", index=False)

print("Archivos guardados correctamente en data/processed:")
print("-", processed_dir / "train_processed.csv")
print("-", processed_dir / "test_processed.csv")

Archivos guardados correctamente en data/processed:
- ..\data\processed\train_processed.csv
- ..\data\processed\test_processed.csv


In [17]:
# Celda 15 - PB-09: guardado del train balanceado
# Objetivo: exportar explícitamente el conjunto de entrenamiento balanceado con SMOTE

df_train_balanced = df_train_processed.copy()

df_train_balanced.to_csv(processed_dir / "train_balanced.csv", index=False)

print("Archivo guardado correctamente en data/processed:")
print("-", processed_dir / "train_balanced.csv")

Archivo guardado correctamente en data/processed:
- ..\data\processed\train_balanced.csv


In [18]:
# Celda 16 - PB-07 / PB-08 / PB-09: persistencia del pipeline
# Objetivo: guardar el pipeline de preprocesamiento y balanceo para reutilizarlo en sprints posteriores

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline_preprocessing, models_dir / "preprocessing_pipeline.pkl")

print("Pipeline guardado correctamente en:")
print("-", models_dir / "preprocessing_pipeline.pkl")

Pipeline guardado correctamente en:
- ..\models\preprocessing_pipeline.pkl


In [19]:
# Celda 17 - PB-07 / PB-08 / PB-09: validación de carga del pipeline persistido
# Objetivo: comprobar que el pipeline guardado puede recuperarse correctamente

pipeline_loaded = joblib.load(models_dir / "preprocessing_pipeline.pkl")

print("Pipeline cargado correctamente.")
print(pipeline_loaded)

Pipeline cargado correctamente.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'BounceRates', 'ExitRates',
                                                   'PageValues', 'SpecialDay',
                                                   'total_paginas',
                                                   'duracion_total']),
                                                 ('cat',
                                                  OneHotEncoder(han

In [20]:
# Celda 18 - PB-07 / PB-08 / PB-09: resumen final del notebook 8
# Objetivo: dejar trazabilidad clara de los outputs generados en el Sprint 2

print("Resumen de artefactos generados:")
print("- ../data/processed/train_processed.csv")
print("- ../data/processed/test_processed.csv")
print("- ../data/processed/train_balanced.csv")
print("- ../models/preprocessing_pipeline.pkl")

print("\nEstado del flujo:")
print("- PB-07 integrado: limpieza base y ajuste de tipos")
print("- PB-08 integrado: features seleccionadas y documentadas")
print("- PB-09 integrado: split estratificado + SMOTE solo en train")
print("- Pipeline validado y persistido correctamente")

Resumen de artefactos generados:
- ../data/processed/train_processed.csv
- ../data/processed/test_processed.csv
- ../data/processed/train_balanced.csv
- ../models/preprocessing_pipeline.pkl

Estado del flujo:
- PB-07 integrado: limpieza base y ajuste de tipos
- PB-08 integrado: features seleccionadas y documentadas
- PB-09 integrado: split estratificado + SMOTE solo en train
- Pipeline validado y persistido correctamente
